# We are using Pharma data and the objective of this project is to identify
# controlled substances(based on dea schedules), high strength drugs, trends in drug marketing timeline, substance classes most associated with regualtion

In [5]:
import pandas as pd
pharma = pd.read_csv('Pharma_data.csv',encoding = 'ISO-8859-1', low_memory = False)

print(claims.columns)
print(claims.head())

Index(['PRODUCTID', 'PRODUCTNDC', 'PRODUCTTYPENAME', 'PROPRIETARYNAME',
       'PROPRIETARYNAMESUFFIX', 'NONPROPRIETARYNAME', 'DOSAGEFORMNAME',
       'ROUTENAME', 'STARTMARKETINGDATE', 'ENDMARKETINGDATE',
       'MARKETINGCATEGORYNAME', 'APPLICATIONNUMBER', 'LABELERNAME',
       'SUBSTANCENAME', 'ACTIVE_NUMERATOR_STRENGTH', 'ACTIVE_INGRED_UNIT',
       'PHARM_CLASSES', 'DEASCHEDULE'],
      dtype='object')
                                        PRODUCTID PRODUCTNDC  \
0  0002-1200_4bd46cbe-cdc1-4329-a8e7-22816bd7fc33  0002-1200   
1  0002-1407_14757f9d-f641-4836-acf3-229265588d1d  0002-1407   
2  0002-1433_aaae85ae-9295-465f-b938-463b74b8d5bd  0002-1433   
3  0002-1434_aaae85ae-9295-465f-b938-463b74b8d5bd  0002-1434   
4  0002-1445_2ecde4c5-5128-4725-9cf5-64384857eb1c  0002-1445   

           PRODUCTTYPENAME      PROPRIETARYNAME PROPRIETARYNAMESUFFIX  \
0  HUMAN PRESCRIPTION DRUG               Amyvid                   NaN   
1  HUMAN PRESCRIPTION DRUG  Quinidine Gluconate           

In [6]:
pharma_sample = pharma.sample(n = 60, random_state = 42)
print(pharma_sample.isnull().sum())

PRODUCTID                    46
PRODUCTNDC                   46
PRODUCTTYPENAME              46
PROPRIETARYNAME              46
PROPRIETARYNAMESUFFIX        57
NONPROPRIETARYNAME           46
DOSAGEFORMNAME               46
ROUTENAME                    47
STARTMARKETINGDATE           46
ENDMARKETINGDATE             59
MARKETINGCATEGORYNAME        46
APPLICATIONNUMBER            48
LABELERNAME                  46
SUBSTANCENAME                47
ACTIVE_NUMERATOR_STRENGTH    47
ACTIVE_INGRED_UNIT           47
PHARM_CLASSES                53
DEASCHEDULE                  58
dtype: int64


In [7]:
pharma_clean = pharma_sample.dropna()

In [11]:
pharma_clean.loc[:, 'STARTMARKETINGDATE'] = pd.to_datetime(
    pharma_clean['STARTMARKETINGDATE'], format='%Y%m%d', errors='coerce')

pharma_clean.loc[:, 'ENDMARKETINGDATE'] = pd.to_datetime(
    pharma_clean['ENDMARKETINGDATE'], format='%Y%m%d', errors='coerce')

pharma_clean.loc[:, 'DEASCHEDULE'] = pharma_clean['DEASCHEDULE'].astype(str).str.strip()

In [12]:
# S-3: Count by DEA Schedule
print(pharma_clean['DEASCHEDULE'].value_counts())

DEASCHEDULE
CII     897
CIV     629
CIII    203
CV       69
Name: count, dtype: int64


In [13]:
total_dea = 1798
print("CII: {:.2f}%".format((897/total_dea)*100))
print("CIII: {:.2f}%".format((203/total_dea)*100))
print("CIV: {:.2f}%".format((629/total_dea)*100))
print("CV: {:.2f}%".format((69/total_dea)*100))

CII: 49.89%
CIII: 11.29%
CIV: 34.98%
CV: 3.84%


In [14]:
# Filter only CII and CIII drugs
high_risk = pharma_clean[pharma_clean['DEASCHEDULE'].isin(['CII', 'CIII'])]

# Count top 10 substances
top_substances = high_risk['SUBSTANCENAME'].value_counts().head(10)
print(top_substances)

SUBSTANCENAME
MORPHINE SULFATE                                                                                                   127
HYDROCODONE BITARTRATE; ACETAMINOPHEN                                                                              108
METHYLPHENIDATE HYDROCHLORIDE                                                                                       81
OXYCODONE HYDROCHLORIDE                                                                                             76
DEXTROAMPHETAMINE SACCHARATE; AMPHETAMINE ASPARTATE MONOHYDRATE; DEXTROAMPHETAMINE SULFATE; AMPHETAMINE SULFATE     59
OXYCODONE HYDROCHLORIDE; ACETAMINOPHEN                                                                              56
HYDROMORPHONE HYDROCHLORIDE                                                                                         54
DEXMETHYLPHENIDATE HYDROCHLORIDE                                                                                    53
OXYMORPHONE HYDROCHLORIDE         

In [21]:
# Cleaning KPI-2 : Avg. strength (mg) of prescribed controlled drugs(Active numerator strength)
# Extract only numeric value before space or slash

import re
# Define a custom function to extract the first numeric value
def extract_strength(val):
    if pd.isnull(val):
        return None
    match = re.search(r'\d+\.?\d*', val)
    return float(match.group()) if match else None

# Apply the function
pharma_clean.loc[:, 'Strength_Value'] = pharma_clean['ACTIVE_NUMERATOR_STRENGTH'].apply(extract_strength)

# Drop nulls in strength + DEA Schedule
strength_df = pharma_clean.dropna(subset=['Strength_Value', 'DEASCHEDULE'])

# Group and calculate average
avg_strength_by_schedule = strength_df.groupby('DEASCHEDULE')['Strength_Value'].mean().sort_values(ascending=False)
print(avg_strength_by_schedule)

DEASCHEDULE
CIII    66.938296
CV      62.289710
CII     44.586227
CIV     32.714547
Name: Strength_Value, dtype: float64


In [23]:
# KPI 3: Number of Drugs Still Marketed Beyond 10 Years

import datetime as dt
today = pd.to_datetime("today")

pharma_clean.loc[:, 'ENDMARKETINGDATE'] = pd.to_datetime(pharma_clean['ENDMARKETINGDATE'], format='%Y%m%d', errors='coerce')
pharma_clean.loc[:, 'STARTMARKETINGDATE'] = pd.to_datetime(pharma_clean['STARTMARKETINGDATE'], format='%Y%m%d', errors='coerce')

# Create column for years marketed
pharma_clean.loc[:, 'Years_Marketed'] = ((pharma_clean['ENDMARKETINGDATE'].fillna(today)) - pharma_clean['STARTMARKETINGDATE']).dt.days / 365.25

# Filter for DEA-scheduled drugs only
dea_drugs = pharma_clean[pharma_clean['DEASCHEDULE'].notnull()]

# Filter for drugs marketed >10 years
long_marketed = dea_drugs[dea_drugs['Years_Marketed'] > 10]

# Count how many
count_10yr_plus = long_marketed.shape[0]
print(f"Controlled drugs marketed for more than 10 years: {count_10yr_plus}")

Controlled drugs marketed for more than 10 years: 1497


In [24]:
# KPI 4: Substances with Multiple pharmalogical classifications
# Filter only controlled substances
dea_drugs = pharma_clean[pharma_clean['DEASCHEDULE'].notnull()]

# Count number of classifications (commas + 1)
dea_drugs['Class_Count'] = dea_drugs['PHARM_CLASSES'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)

# Filter drugs with multiple classes
multi_class = dea_drugs[dea_drugs['Class_Count'] > 1]

# Count how many
multi_class_count = multi_class.shape[0]
print(f"Controlled drugs with multiple pharmacological classifications: {multi_class_count}")

Controlled drugs with multiple pharmacological classifications: 1672


In [25]:
pharma_clean.to_excel("pharma_clean.xlsx", index=False)

In [27]:
# Count classes per row
pharma_clean.loc[:, 'Class_Count'] = pharma_clean['PHARM_CLASSES'].fillna('').apply(lambda x: len(x.split(',')))

# Save to Excel
pharma_clean.to_excel("pharma_clean.xlsx", index=False)